# Cohort Analysis of Contributors (8Knot issue #1110)

**Question this notebook answers.** People join a project in different months. If we group contributors by the month of their first action, how many of each group are still active 1, 2, 3 or more months later?

**How it works, in plain steps**
1. Pick the repos and the date range in the Settings cell.
2. Pull every contributor action (commits, issues opened, pull requests opened, pull request reviews, comments) from the CollectOSS database. Bot accounts are left out (see EXCLUDE_GH_TYPES in the Settings cell).
3. Find each contributor's first action in the selected repos. That month is their cohort.
4. Count how many people from each cohort are active in each month after that.
5. Show the result as a table and a heatmap.

**One limit to know about.** A contributor whose first action in these repos happened before the start date would wrongly look new if we only looked inside the date range. So the notebook checks all earlier history first and marks those people as "started earlier". They are left out of the cohorts and counted separately, so they cannot distort the result.

## 1. Install the tools (run once)

In [ ]:
# Only needed the first time. Remove the # to run it.
# %pip install pandas sqlalchemy psycopg2-binary matplotlib

## 2. Settings
Change the values in this cell, then run the cells below it in order.

In [ ]:
# --- Database connection (the sample database from the CollectOSS docs) ---
DB_HOST = "localhost"
DB_PORT = 5433
DB_NAME = "sample_collected_data"
DB_USER = "sample_user"
DB_PASSWORD = "sample_password"

# --- Which repos? ---
# Use None for all repos, or a list of repo ids like [1, 2, 3].
# Run the "List the repos" cell below to see the ids.
REPO_IDS = None

# --- Which dates? (year-month-day) ---
START_DATE = "2023-01-01"
END_DATE = "2024-12-31"

# --- How many months after the first action to show ---
MAX_MONTHS = 12

# --- Which account types to leave out? ---
# The database marks bot accounts with the type "Bot". Add "Organization" to the list to leave those out too.
# Use an empty list [] to keep everyone.
EXCLUDE_GH_TYPES = ["Bot"]


## 3. Connect to the database

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sqlalchemy import create_engine, text

url = f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(url)

def run(sql):
    # Small helper. Sends a query to the database and returns a table.
    with engine.connect() as conn:
        return pd.read_sql(text(sql), conn)

print(run("select current_database() as database, now() as time_now"))

## 4. List the repos (so you can choose ids)

In [ ]:
repos = run("""
    select repo_id, repo_name, repo_git
    from data.repo
    order by repo_id
""")
print(len(repos), "repos in the database")
repos.head(20)

## 5. Pull the contributor actions
Each query below reads one kind of action. For each one we group by person and month, so the result stays small.

In every table the contributor id column has a different name, so we rename it to `contributor` and turn it into text so all five line up.

In [ ]:
# This builds the repo filter line. With REPO_IDS = None there is no filter.
if REPO_IDS is None:
    repo_filter = ""
else:
    ids = ", ".join(str(int(i)) for i in REPO_IDS)
    repo_filter = f"and repo_id in ({ids})"

sql = f"""
select contributor, action_month, action_type, count(*) as n
from (
    -- 1. Commits (the commit author)
    select cmt_ght_author_id::text as contributor,
           date_trunc('month', coalesce(cmt_author_timestamp, cmt_committer_timestamp))::date as action_month,
           'commit' as action_type
    from data.commits
    where cmt_ght_author_id is not null {repo_filter}

    union all
    -- 2. Issues opened (the issue author is in reporter_id; rows that are really pull requests are left out)
    select coalesce(reporter_id, cntrb_id)::text,
           date_trunc('month', created_at)::date,
           'issue opened'
    from data.issues
    where pull_request is null and coalesce(reporter_id, cntrb_id) is not null {repo_filter}

    union all
    -- 3. Pull requests opened
    select pr_augur_contributor_id::text,
           date_trunc('month', pr_created_at)::date,
           'pull request opened'
    from data.pull_requests
    where pr_augur_contributor_id is not null {repo_filter}

    union all
    -- 4. Pull request reviews
    select cntrb_id::text,
           date_trunc('month', pr_review_submitted_at)::date,
           'pull request review'
    from data.pull_request_reviews
    where cntrb_id is not null {repo_filter}

    union all
    -- 5. Comments and messages
    select cntrb_id::text,
           date_trunc('month', msg_timestamp)::date,
           'comment'
    from data.message
    where cntrb_id is not null {repo_filter}
) actions
where action_month is not null
group by contributor, action_month, action_type
"""

actions = run(sql)
actions["action_month"] = pd.to_datetime(actions["action_month"])

# Leave out bot accounts. The contributors table has a gh_type column (User, Bot, Organization).
if EXCLUDE_GH_TYPES:
    types = ", ".join(f"'{t}'" for t in EXCLUDE_GH_TYPES)
    excluded = run(f"""
        select cntrb_id::text as contributor
        from data.contributors
        where gh_type in ({types})
    """)
    before = actions["contributor"].nunique()
    actions = actions[~actions["contributor"].isin(excluded["contributor"])]
    after = actions["contributor"].nunique()
    print(before - after, "accounts of type", EXCLUDE_GH_TYPES, "left out")
print(len(actions), "rows (person, month, action type)")
actions.head()

### Quick data check
If one action type shows zero rows, its contributor link may be missing in this database. That is useful to know before trusting the result.

In [ ]:
print(actions.groupby("action_type")["contributor"].nunique().rename("people"))
print()
print("People overall", actions["contributor"].nunique())
print("Earliest month in data", actions["action_month"].min().date())
print("Latest month in data  ", actions["action_month"].max().date())

## 6. Find each person's first action
We look at all history in the selected repos, not only the date range. That lets us spot people who started before the start date.

In [ ]:
first_month = actions.groupby("contributor")["action_month"].min().rename("first_month")

start = pd.Timestamp(START_DATE).to_period("M").to_timestamp()
end = pd.Timestamp(END_DATE).to_period("M").to_timestamp()

people = first_month.reset_index()
people["started_earlier"] = people["first_month"] < start

print("People with a first action before the start date (started earlier)", int(people["started_earlier"].sum()))
print("People whose first action is inside the range", int(((people["first_month"] >= start) & (people["first_month"] <= end)).sum()))

## 7. Build the cohorts
A cohort is the month of a person's first action. For every later month we ask whether that person did anything.

In [ ]:
# Keep only actions inside the date range
in_range = actions[(actions["action_month"] >= start) & (actions["action_month"] <= end)]

# One row per person per active month
active = in_range[["contributor", "action_month"]].drop_duplicates()
active = active.merge(people, on="contributor")

# People who started earlier are left out of the cohorts
active = active[~active["started_earlier"]].copy()
active["cohort"] = active["first_month"]

# How many months after the cohort month is this activity?
active["months_after"] = (
    (active["action_month"].dt.year - active["first_month"].dt.year) * 12
    + (active["action_month"].dt.month - active["first_month"].dt.month)
)

active = active[(active["months_after"] >= 0) & (active["months_after"] <= MAX_MONTHS)]

counts = (
    active.groupby(["cohort", "months_after"])["contributor"]
    .nunique()
    .unstack("months_after")
    .fillna(0)
    .astype(int)
)
counts.index = [d.strftime("%Y-%m") for d in counts.index]
counts.index.name = "Cohort (month of first action)"
counts

## 8. Turn counts into percentages
Month 0 is the cohort month, so it is always 100 percent. Each later column shows what share of the cohort was still active.

In [ ]:
cohort_size = counts[0]
retention = counts.divide(cohort_size, axis=0) * 100
retention = retention.round(1)

# Months that have not happened yet (after the end date) are left empty, not shown as 0
end_period = end.to_period("M")
for cohort_label in retention.index:
    cohort_period = pd.Period(cohort_label, freq="M")
    months_possible = (end_period - cohort_period).n
    for m in retention.columns:
        if m > months_possible:
            retention.loc[cohort_label, m] = float("nan")
retention.insert(0, "People", cohort_size)
retention

## 9. Heatmap

In [ ]:
data = retention.drop(columns="People")

fig, ax = plt.subplots(figsize=(11, max(3, 0.5 * len(data) + 1.5)))
image = ax.imshow(data.values, cmap="Blues", vmin=0, vmax=100, aspect="auto")

ax.set_xticks(range(len(data.columns)))
ax.set_xticklabels(data.columns)
ax.set_yticks(range(len(data.index)))
ax.set_yticklabels([f"{c}  (n={retention.loc[c, 'People']})" for c in data.index])
ax.set_xlabel("Months after first action")
ax.set_ylabel("Cohort")
ax.set_title("Contributor retention by cohort (percent still active)")

# Write the percentage in each box. Empty boxes are months that have not happened yet.
for r in range(data.shape[0]):
    for c in range(data.shape[1]):
        value = data.values[r, c]
        if not pd.isna(value):
            ax.text(c, r, f"{value:.0f}", ha="center", va="center",
                    color="white" if value > 55 else "black", fontsize=8)

fig.colorbar(image, ax=ax, label="Percent of cohort")
plt.tight_layout()
plt.show()

## 10. How to read it
1. Each row is a group of people who made their first action in the same month.
2. Each column is how many months later we check.
3. A box shows the share of that group that did something again in that month.
4. A row that fades quickly means people tried the project once and left. A row that stays dark means people stayed.

**Limits to remember**
1. Cohorts near the end of the date range have fewer months to show, so their boxes at the right stay empty.
2. The notebook only counts people the database could link to a contributor id.
3. A person who started before the start date is marked "started earlier" and left out.
4. Bots are found by the gh_type column in the contributors table. A bot that the database labels as a normal user would still be counted.
